In [17]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from src.models.autoencoder import Autoencoder
from src.data.load_cifar10 import get_cifar10_loaders


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [5]:
# Dane
train_loader, val_loader, test_loader = get_cifar10_loaders(batch_size=64)


In [6]:
# Model
model = Autoencoder(latent_dim=256).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

https://www.datacamp.com/tutorial/pytorch-cnn-tutorial

In [18]:
# Trening

BASE_DIR = os.getcwd()
save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'autoencoder', 'cifar10')
os.makedirs(save_dir, exist_ok=True)


num_epochs = 2

train_losses = []
val_losses = []
epoch_number = 0
best_val_loss = float('inf')
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for batch_idx, (image, _) in enumerate(train_loader):
        image = image.to(device)
        # target = target.to(device)

        outputs = model(image)
        loss = criterion(outputs, image)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f"  [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(train_loader)} "
                  f"Loss: {loss.item():.4f}")
    # Średni train loss
    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    # validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for image, _ in val_loader:
            image = image.to(device)
            # target = target.to(device)

            outputs = model(image)
            loss = criterion(outputs, image)
            val_loss += loss.item()
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")


    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'latent_dim': 256,
            'train_losses': train_losses,
            'val_losses': val_losses
        }
        torch.save(checkpoint, os.path.join(save_dir, 'autoencoder_cifar10.pt'))


df = pd.DataFrame({
    'epoch': range(1, num_epochs + 1),
    'train_loss': train_losses,
    'val_loss': val_losses
})

history_csv = os.path.join(save_dir, 'autoencoder_cifar10_training_results.csv')
df.to_csv(history_csv, index=False)


  [1/2] Batch 0/625 Loss: 0.0214
  [1/2] Batch 100/625 Loss: 0.0216
  [1/2] Batch 200/625 Loss: 0.0240
  [1/2] Batch 300/625 Loss: 0.0210
  [1/2] Batch 400/625 Loss: 0.0230
  [1/2] Batch 500/625 Loss: 0.0190
  [1/2] Batch 600/625 Loss: 0.0220
Epoch [1/2]
Train Loss: 0.0215
Val Loss:   0.0213
  [2/2] Batch 0/625 Loss: 0.0206
  [2/2] Batch 100/625 Loss: 0.0208
  [2/2] Batch 200/625 Loss: 0.0211
  [2/2] Batch 300/625 Loss: 0.0214
  [2/2] Batch 400/625 Loss: 0.0214
  [2/2] Batch 500/625 Loss: 0.0218
  [2/2] Batch 600/625 Loss: 0.0198
Epoch [2/2]
Train Loss: 0.0215
Val Loss:   0.0216
